# Browser CNN

Second model for the browser, replacing Teachable Machine. Small CNN on log-mel
spectrograms of one second of audio.

Features come from `feature_extraction/logmel.py`, the same module the parity
check compared against the JavaScript. Nothing here redefines them.

Trains on `training/` + `augmented_training/`, validates on `validation/`.
`testing/` is not touched anywhere in this notebook.

In [ ]:
!pip install -q tensorflowjs

print()
print("If Colab shows a 'Restart runtime' button, click it, then carry on from")
print("the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError('PROJECT_PATH does not exist: ' + PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# The features must come from the project module. If this import does not
# resolve we stop, rather than quietly training on a second definition.
try:
    from feature_extraction import logmel
    from src import browser_cnn
except Exception as e:
    raise RuntimeError('could not import the project modules from ' + PROJECT_PATH) from e

if not os.path.abspath(logmel.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError('logmel.py came from the wrong place: ' + logmel.__file__)

print('logmel     ', logmel.__file__)
print('browser_cnn', browser_cnn.__file__)
print('shape      ', (logmel.FRAMES, logmel.MELS, 1))
print('sr', logmel.SR, 'frame', logmel.FRAME, 'step', logmel.STEP, 'mels', logmel.MELS)

In [ ]:
names = browser_cnn.labels()
print(len(names), 'labels in encoder order:')
for i, n in enumerate(names):
    print(' ', i, n)

In [ ]:
# Windows are cached to data/metadata/ so a rerun does not redo the audio.
xt, yt = browser_cnn.get('train', browser_cnn.TRAIN_DIRS, names)
xv, yv = browser_cnn.get('val', browser_cnn.VAL_DIRS, names)

print()
print('train', xt.shape, 'val', xv.shape)

import numpy as np
print('train per class', np.bincount(yt, minlength=len(names)))
print('val   per class', np.bincount(yv, minlength=len(names)))

In [ ]:
# Class weights are applied inside train(), so a lagging class is not ignored.
print(browser_cnn.class_weights(yt, len(names)))

In [ ]:
model, history = browser_cnn.train(xt, yt, xv, yv, len(names))
model.summary()

In [ ]:
import json
import os

import matplotlib.pyplot as plt

h = history.history

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(h['loss'], label='train')
ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoch'); ax[0].legend()

ax[1].plot(h['accuracy'], label='train')
ax[1].plot(h['val_accuracy'], label='val')
ax[1].set_title('accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()

ax[2].plot(h['val_macro_f1'])
ax[2].set_title('val macro f1'); ax[2].set_xlabel('epoch')

plt.tight_layout()

out = os.path.join(PROJECT_PATH, 'reports', 'plots')
os.makedirs(out, exist_ok=True)
plt.savefig(os.path.join(out, 'browser_cnn_curves.png'), dpi=120)
plt.show()

with open(os.path.join(PROJECT_PATH, 'reports', 'browser_cnn_history.json'), 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in h.items()}, f, indent=2)

best = max(h['val_macro_f1'])
print('best val macro f1', round(best, 4), 'at epoch', h['val_macro_f1'].index(best) + 1)
print('epochs run', len(h['loss']))

In [ ]:
# Export into gtm_model/, the same folder and the same three artefact kinds the
# Teachable Machine export produced.
import glob
import json
import os
import shutil

import tensorflowjs as tfjs

target = os.path.join(PROJECT_PATH, 'gtm_model')
os.makedirs(target, exist_ok=True)

for old in glob.glob(os.path.join(target, '*.bin')) + \
           glob.glob(os.path.join(target, 'model.json')) + \
           glob.glob(os.path.join(target, 'metadata.json')):
    os.remove(old)
    print('removed old', os.path.basename(old))

tfjs.converters.save_keras_model(model, target)

meta = {
    'version': 'browser-cnn-v1',
    'labels': names,
    'features': {
        'sample_rate': logmel.SR,
        'window': logmel.WIN,
        'pad_to': logmel.PAD,
        'frame_length': logmel.FRAME,
        'frame_step': logmel.STEP,
        'fft_length': logmel.FFT,
        'mel_bins': logmel.MELS,
        'fmin': logmel.FMIN,
        'fmax': logmel.FMAX,
        'log_offset': logmel.EPS,
        'frames': logmel.FRAMES,
        'normalise': 'per example: subtract mean, divide by std',
        'input_shape': [logmel.FRAMES, logmel.MELS, 1],
    },
}

with open(os.path.join(target, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

for f in sorted(os.listdir(target)):
    p = os.path.join(target, f)
    print(f, round(os.path.getsize(p) / 1024, 1), 'KB')

## Bringing it back

`gtm_model/` on Drive now holds `model.json`, the weight shard, and
`metadata.json`. Copy that folder into the project locally, over the old
Teachable Machine export.

`reports/plots/browser_cnn_curves.png` and `reports/browser_cnn_history.json`
hold the curves.

The evaluation over `testing/` runs locally, not here.